# 📊 Trabalho Final: Redes Neurais Profundas
**Universidade Federal de Goiás (UFG) — Instituto de Informática (INF)**

**Projeto:** Análise Arquitetural do Congelamento de Camadas na Mitigação de *Domain Shift* e *Language Shift* em Transformers Multilíngues

**Equipe:**
- Geovana Teixeira Camargo — 202303338
- Sebastião Corrêa Fraga Neto — 202301487
- Pedro Reis Pimenta — 202303359
- Henrique Matheus Mendonça de Miranda — 202402479

---
> **Status deste notebook:** **Etapa 1 — Preparação de Dados e Auditoria Semântica** implementada e executável de ponta a ponta no Google Colab (Tasks 1.1, 1.2 e 1.3 do plano de execução). As Etapas 2–4 (modelagem, treino e avaliação) serão adicionadas nas próximas entregas.

## 1. Definição do Problema (*Research Question*)

### 1.1 Contexto e Lacuna (*The Gap*)
Em tarefas de NLP, modelos pré-treinados em múltiplos idiomas (como o **XLM-RoBERTa**) são eficazes, mas sofrem degradação de performance quando o **domínio** (ex.: de Eletrônicos para Beleza/Saúde) ou o **idioma** (ex.: de Inglês para Português) muda entre o treino e o teste (*Zero-shot cross-lingual*).

A lacuna reside em entender como o **congelamento seletivo de camadas** (inicial vs. final) pode preservar o conhecimento linguístico e semântico para mitigar essas quedas de performance.

### 1.2 Pergunta de Pesquisa
> *"Em tarefas de classificação binária de sentimento, qual estratégia de congelamento seletivo de pesos em um XLM-RoBERTa mitiga mais eficazmente a degradação de F1-macro causada por Language Shift versus Domain Shift?"*

### 1.3 Hipóteses
- **H1 (Language Shift):** Congelar camadas iniciais (0–5) mitiga a queda entre Inglês e Português.
- **H2 (Domain Shift):** Congelar camadas finais (6–11) mitiga a queda entre os domínios de Eletrônicos e Beleza/Saúde.

## 2. Métodos (Arquitetura e Datasets)

Esta seção detalha o delineamento experimental construído para investigar as causas da degradação de performance em modelos de PLN quando submetidos a cenários reais de transferência *Zero-Shot Cross-Lingual*. A abordagem foca na manipulação estrutural da arquitetura base (**XLM-RoBERTa**) durante o *fine-tuning*, buscando isolar e quantificar o impacto de duas variáveis independentes: a alteração do idioma (*Language Shift*) e a mudança da semântica da tarefa (*Domain Shift*).

### 2.1 Arquitetura do Modelo
Utilizaremos o **XLM-RoBERTa-base** (~278M parâmetros) com uma *Classification Head* (Dropout + Linear) para classificação binária (Positivo/Negativo).

---
## 🟦 Etapa 1 — Preparação de Dados e Auditoria Semântica

Esta etapa implementa as três *tasks* do plano de execução:

| Task | Descrição | Saída |
|------|-----------|-------|
| **1.1** | Download e exploração inicial dos datasets (Amazon EN, B2W PT) | Pools carregados + EDA |
| **1.2** | Filtragem por domínio (EN: keyword · PT: categoria) → partições **S1, S2, S3, S4** | Subconjuntos sem interseção de IDs |
| **1.3** | Auditoria de qualidade via LLM zero-shot (100 amostras/subconjunto) | Tabela de precisão (≥ 80%) |

**Subconjuntos-alvo** (Seção 5.4 da Metodologia):

| ID | Origem | Idioma | Domínio | Função |
|----|--------|--------|---------|--------|
| **S1** | Amazon (keyword) | EN | Eletrônicos | Treino (80%) + Validação (20%) |
| **S2** | Amazon (keyword) | EN | Beleza | Teste — isola *Domain Shift* |
| **S3** | B2W (categoria) | PT | Eletrônicos | Teste — isola *Language Shift* |
| **S4** | B2W (categoria) | PT | Beleza | Teste — impacto combinado |

**Esquema de rótulos** (Seção 4): estrelas 1–2 → **Negativo (0)**, 4–5 → **Positivo (1)**, nota 3 descartada.

### 2.0 Configuração do Ambiente

In [ ]:
# 1. Instalação das bibliotecas necessárias
#    (datasets<3.0.0 mantém compatibilidade com leitura via streaming usada aqui)
!pip install -q "transformers" "datasets<3.0.0" evaluate accelerate scikit-learn matplotlib seaborn
print("Bibliotecas instaladas.")

In [ ]:
# 2. Hardware e Fixação de Seeds (Seção 10 da Metodologia)
import torch, random, os
import numpy as np
from transformers import set_seed
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware em uso: {device}")

SEEDS = [42, 123, 2024]   # seeds oficiais do projeto (usadas nas Etapas 3-4)
SEED_DATA = 42            # seed da preparação de dados (amostragem / balanceamento)

def fixar_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    set_seed(seed)

fixar_seed(SEED_DATA)
print(f"Seeds do projeto: {SEEDS} | Seed da preparação de dados: {SEED_DATA}")

In [ ]:
# 3. Parâmetros centrais da Etapa 1
from pathlib import Path

DIR_SAIDA = Path("/content/data_processed")
DIR_SAIDA.mkdir(parents=True, exist_ok=True)

# Mapeamento estrela -> sentimento binário (Seção 4): 1,2->0(Neg) | 4,5->1(Pos) | 3 descartado
def estrela_para_sentimento(nota):
    try:
        n = float(nota)
    except (TypeError, ValueError):
        return None
    if n <= 2: return 0
    if n >= 4: return 1
    return None  # nota 3 (neutra) descartada

MAX_CANDIDATOS_EN = 20000      # candidatos coletados por domínio EN (antes do balanceamento)
MAX_LINHAS_STREAM_EN = 600000  # teto de linhas varridas no stream do Amazon
N_AUDITORIA = 100              # amostras por subconjunto para auditoria manual (Task 1.3)
LIMIAR_PRECISAO = 0.80         # critério de aceitação do filtro
print("Configurações carregadas.")

### 2.2 Definição de Domínios — EN por palavra-chave, PT por categoria

A atribuição de domínio usa **dois métodos**, um por base, conforme a disponibilidade de metadado:

- **Inglês (Amazon `amazon_polarity`)** — não traz categoria de produto, então usamos **filtro lexical**: a avaliação pertence a um domínio se contém ≥1 palavra-chave dele **e nenhuma** do outro (descarta ambíguos/neutros).
- **Português (B2W)** — traz `site_category_lv1`, então usamos **a categoria do produto** diretamente (ground-truth, precisão ~100%), eliminando o viés do filtro lexical justamente no lado mais escasso.

Essa assimetria é proposital: aproveita o metadado onde ele existe (PT) e mantém o método lexical onde não existe (EN). A auditoria zero-shot (Task 1.3) valida a precisão do lado EN; o lado PT é exato por construção.

> **Por que isso importa:** o filtro por keyword em PT capturava só ~300 avaliações negativas de beleza; a categoria `Beleza e Perfumaria` tem **2.372** (descoberto na Etapa 0). O gargalo era o método, não o domínio.

In [ ]:
# Definição dos domínios — Eletrônicos × Beleza
# Abordagem híbrida (declarada na metodologia):
#   • EN (amazon_polarity, SEM metadado de categoria) -> filtro por PALAVRA-CHAVE.
#   • PT (B2W, COM site_category_lv1)                -> filtro por CATEGORIA (ground-truth).

# --- EN: palavras-chave INEQUÍVOCAS de eletrônicos ---
# Removidas as ambíguas (screen, camera, display, speaker, keyboard, processor): elas
# vazavam para reviews de filme/música/cozinha ("on screen", "camera work", "food processor")
# e derrubavam a precisão do filtro EN. Mantidos só termos específicos de eletrônica de consumo.
keywords_eletronicos_en = ['battery','usb','charger','charging','wifi','bluetooth',
                           'smartphone','laptop','tablet','touchscreen','headphone',
                           'headphones','earbuds','smartwatch','phone','phones','router','hdmi']
keywords_beleza_en      = ['skin','scent','fragrance','perfume','cream',
                           'lotion','shampoo','conditioner','moisturizer','makeup',
                           'cosmetic','serum','sunscreen','soap','wrinkle']
KEYWORDS = {'en': {'eletronicos': keywords_eletronicos_en, 'beleza': keywords_beleza_en}}

# --- PT: mapeamento de categoria (site_category_lv1) -> domínio ---
# Eletrônicos = cluster de eletrônica de consumo (parear com a definição ampla do EN).
# Beleza      = Beleza e Perfumaria. Demais categorias -> None (descartadas).
CATEGORIAS_PT = {
    'Celulares e Smartphones':  'eletronicos',
    'Informática e Acessórios': 'eletronicos',
    'TV e Home Theater':        'eletronicos',
    'Beleza e Perfumaria':      'beleza',
}

def categoria_para_dominio(cat):
    """Mapeia site_category_lv1 -> 'eletronicos' | 'beleza' | None."""
    if not isinstance(cat, str):
        return None
    return CATEGORIAS_PT.get(cat.strip(), None)

print("Domínios: EN por keyword (lista enxuta/inequívoca) | PT por categoria (site_category_lv1).")
print("Categorias PT consideradas:", CATEGORIAS_PT)

In [ ]:
# Filtro lexical (usado SÓ no lado EN; o PT usa categoria)
import re

def _compilar_regex(lista_kw):
    partes = [re.escape(kw.lower()) for kw in lista_kw]
    return re.compile(r"\b(?:" + "|".join(partes) + r")\b", re.UNICODE)

REGEX = {lang: {dom: _compilar_regex(kws) for dom, kws in doms.items()}
         for lang, doms in KEYWORDS.items()}

def classificar_dominio(texto, lang='en'):
    # Regra 5.2 (EN): >=1 keyword do domínio E nenhuma do outro
    if not isinstance(texto, str) or not texto.strip():
        return None
    t = texto.lower()
    tem_eletr  = bool(REGEX[lang]['eletronicos'].search(t))
    tem_beleza = bool(REGEX[lang]['beleza'].search(t))
    if tem_eletr and not tem_beleza: return 'eletronicos'
    if tem_beleza and not tem_eletr: return 'beleza'
    return None

# Sanity checks — EN (keyword)
assert classificar_dominio("the battery life is amazing", 'en') == 'eletronicos'
assert classificar_dominio("this moisturizer cream is great for my skin", 'en') == 'beleza'
assert classificar_dominio("the laptop smells like perfume", 'en') is None
# Sanity checks — PT (categoria)
assert categoria_para_dominio('Celulares e Smartphones') == 'eletronicos'
assert categoria_para_dominio('Beleza e Perfumaria') == 'beleza'
assert categoria_para_dominio('Livros') is None
print("Classificação validada: EN por keyword, PT por categoria.")

### 2.3 Task 1.1 — Carregamento dos Datasets

**Inglês (Amazon Customer Reviews).** Usamos o `amazon_polarity` (Hugging Face) em modo *streaming* — ele já vem com o rótulo binário resultante do mesmo mapeamento de estrelas (1–2 → Negativo, 4–5 → Positivo, 3 descartado), e o streaming evita baixar os ~1,6 GB completos. Coletamos candidatos enquanto aplicamos o filtro lexical de domínio.

In [ ]:
# Task 1.1 (EN) — Amazon via Hugging Face (streaming) + filtro de domínio on-the-fly
from datasets import load_dataset
import pandas as pd

def carregar_amazon_en():
    erros = []
    for repo in ["fancyzhx/amazon_polarity", "amazon_polarity"]:
        try:
            return load_dataset(repo, split="train", streaming=True)
        except Exception as e:
            erros.append(f"{repo}: {e}")
    raise RuntimeError("Falha ao carregar Amazon EN:\n" + "\n".join(erros))

stream_en = carregar_amazon_en()

coletado = {'eletronicos': [], 'beleza': []}
vistos = 0
for ex in stream_en:
    vistos += 1
    texto = ((ex.get('title') or '') + '. ' + (ex.get('content') or '')).strip()
    dom = classificar_dominio(texto, 'en')
    if dom is not None and len(coletado[dom]) < MAX_CANDIDATOS_EN:
        coletado[dom].append({'id': f"amz_{vistos}", 'texto': texto,
                              'label': int(ex['label']), 'idioma': 'en', 'dominio': dom})
    if vistos >= MAX_LINHAS_STREAM_EN: break
    if len(coletado['eletronicos']) >= MAX_CANDIDATOS_EN and len(coletado['beleza']) >= MAX_CANDIDATOS_EN: break

df_en = pd.DataFrame(coletado['eletronicos'] + coletado['beleza'])
print(f"Linhas Amazon varridas no stream: {vistos:,}")
print(f"Candidatos EN -> Eletrônicos: {len(coletado['eletronicos']):,} | Beleza: {len(coletado['beleza']):,}")
display(df_en.head())

**Português (B2W Reviews).** Lê o arquivo `/content/b2w.csv` (faça o upload antes); se não existir, baixa do repositório oficial. O domínio em PT vem da coluna **`site_category_lv1`** (categoria do produto) — não de palavras-chave.

> **Olist foi descartado** nesta versão: o dataset de avaliações do Olist não traz a categoria do produto no nível da avaliação (exigiria *join* multi-tabela), e o B2W sozinho, filtrado por categoria, já excede as metas de tamanho (Beleza e Perfumaria: 2.372 negativos; cluster de eletrônicos: ~8.500).

In [ ]:
# Task 1.1 (PT) — B2W Reviews (arquivo enviado ou download automático)
import pandas as pd, os

CAMINHO_B2W = "/content/b2w.csv"
URL_B2W = "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv"

def carregar_b2w():
    if os.path.exists(CAMINHO_B2W):
        print(f"Lendo B2W do arquivo enviado: {CAMINHO_B2W}")
        return pd.read_csv(CAMINHO_B2W, low_memory=False)
    print("Arquivo /content/b2w.csv não encontrado. Tentando download automático do repositório oficial...")
    df = pd.read_csv(URL_B2W, low_memory=False)
    df.to_csv(CAMINHO_B2W, index=False)
    print("Download concluído e salvo em /content/b2w.csv")
    return df

df_b2w = carregar_b2w()
print(f"B2W carregado: {df_b2w.shape[0]:,} linhas, {df_b2w.shape[1]} colunas.")
print("Colunas:", list(df_b2w.columns))
display(df_b2w.head(3))

In [ ]:
# Normalização do pool PT (B2W) -> [id, texto, nota, label, idioma, categoria]
# O domínio em PT virá da CATEGORIA (site_category_lv1), atribuída na célula de particionamento.
import pandas as pd

def _col(df, *nomes):
    for n in nomes:
        if n in df.columns: return n
    return None

def _serie(df, nome):
    n = len(df)
    return df[nome].fillna('').astype(str) if nome else pd.Series(['']*n)

df_pt = pd.DataFrame()
titulo = _serie(df_b2w, _col(df_b2w, 'review_title'))
texto  = _serie(df_b2w, _col(df_b2w, 'review_text'))
df_pt['texto'] = (titulo + '. ' + texto).str.strip()

c_nota = _col(df_b2w, 'overall_rating')
df_pt['nota'] = pd.to_numeric(df_b2w[c_nota], errors='coerce') if c_nota else None

c_cat = _col(df_b2w, 'site_category_lv1')
df_pt['categoria'] = _serie(df_b2w, c_cat)

df_pt['label'] = df_pt['nota'].apply(estrela_para_sentimento)   # estrela -> binário
df_pt = df_pt[df_pt['texto'].str.len() > 0]                     # remove vazios
df_pt = df_pt.dropna(subset=['label']).copy()                  # remove nota 3 / inválidos
df_pt['label']  = df_pt['label'].astype(int)
df_pt['idioma'] = 'pt'
df_pt['id']     = ['pt_' + str(i) for i in range(len(df_pt))]

print(f"Pool PT (B2W) após mapeamento binário: {len(df_pt):,} avaliações (sem nota 3 / sem texto vazio).")
print("Maiores categorias (site_category_lv1):")
print(df_pt['categoria'].value_counts().head(8).to_string())
display(df_pt[['id','categoria','nota','label','texto']].head(3))

### 2.4 Task 1.2 — Particionamento por Domínio (S1–S4)

In [ ]:
# Task 1.2 — domínio do PT pela CATEGORIA; EN já recebeu domínio por keyword no carregamento
df_pt['dominio'] = df_pt['categoria'].apply(categoria_para_dominio)

S1_raw = df_en[df_en['dominio'] == 'eletronicos'].copy()  # EN Eletrônicos (keyword)
S2_raw = df_en[df_en['dominio'] == 'beleza'].copy()        # EN Beleza (keyword)
S3_raw = df_pt[df_pt['dominio'] == 'eletronicos'].copy()  # PT Eletrônicos (categoria)
S4_raw = df_pt[df_pt['dominio'] == 'beleza'].copy()        # PT Beleza (categoria)

for nome, S in [('S1 (EN/Eletrônicos)', S1_raw), ('S2 (EN/Beleza)', S2_raw),
                ('S3 (PT/Eletrônicos)', S3_raw), ('S4 (PT/Beleza)', S4_raw)]:
    pos = int((S['label']==1).sum()); neg = int((S['label']==0).sum())
    print(f"{nome:>22}: {len(S):>7,} | Positivos: {pos:>6,} | Negativos: {neg:>6,}")

In [ ]:
# VERIFY (Task 1.2) — as partições não compartilham IDs
import itertools
ids = {'S1': set(S1_raw['id']), 'S2': set(S2_raw['id']),
       'S3': set(S3_raw['id']), 'S4': set(S4_raw['id'])}
ok = True
for a, b in itertools.combinations(ids, 2):
    inter = ids[a] & ids[b]
    if inter:
        ok = False; print(f"❌ Interseção entre {a} e {b}: {len(inter)} IDs")
assert ok, "Há interseção de IDs entre partições!"
print("✅ Nenhuma interseção de IDs entre S1, S2, S3 e S4.")

### 2.5 Exploração dos Dados (EDA)

In [ ]:
# EDA: distribuição de classes por subconjunto e tamanho dos textos
import matplotlib.pyplot as plt
import numpy as np

subs = {'S1\nEN/Eletr': S1_raw, 'S2\nEN/Belez': S2_raw, 'S3\nPT/Eletr': S3_raw, 'S4\nPT/Belez': S4_raw}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

labels = list(subs.keys())
pos = [int((S['label']==1).sum()) for S in subs.values()]
neg = [int((S['label']==0).sum()) for S in subs.values()]
x = np.arange(len(labels))
axes[0].bar(x-0.2, neg, 0.4, label='Negativo')
axes[0].bar(x+0.2, pos, 0.4, label='Positivo')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_title('Distribuição de classes por subconjunto (bruto)')
axes[0].set_ylabel('nº de avaliações'); axes[0].legend()

for nome, S in subs.items():
    comp = S['texto'].str.split().apply(len)
    axes[1].hist(comp, bins=40, alpha=0.5, label=nome.replace('\n', '/'))
axes[1].set_title('Tamanho dos textos (nº de palavras)')
axes[1].set_xlabel('palavras'); axes[1].set_xlim(0, 120); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# DIAGNÓSTICO (antes de balancear) — contagem POR CLASSE de cada compartimento.
# O balanceamento sempre é limitado pela MENOR contagem de uma classe entre os
# compartimentos — historicamente a classe NEGATIVA do lado PT (avaliações negativas
# são minoria, e mais ainda em domínios de nicho). Esta célula expõe esse gargalo
# explicitamente, para você confirmar com os próprios olhos ao rodar no Colab.

print("Contagem por classe ANTES do balanceamento (neg = 0, pos = 1):\n")
print(f"{'Compartimento':>22} | {'Negativos':>9} | {'Positivos':>9} | {'min(classe)':>11}")
print("-" * 62)

diag = []
for nome, S in [('S1 (EN/Eletrônicos)', S1_raw), ('S2 (EN/Beleza)', S2_raw),
                ('S3 (PT/Eletrônicos)', S3_raw), ('S4 (PT/Beleza)', S4_raw)]:
    neg = int((S['label'] == 0).sum())
    pos = int((S['label'] == 1).sum())
    mc  = min(neg, pos)
    diag.append((nome, neg, pos, mc))
    print(f"{nome:>22} | {neg:>9,} | {pos:>9,} | {mc:>11,}")
print("-" * 62)

gargalo = min(diag, key=lambda r: r[3])
print(f"\n🔎 Gargalo global (menor classe entre os compartimentos): {gargalo[0]} "
      f"→ apenas {gargalo[3]:,} exemplos na classe minoritária.")
print("   Com o balanceamento DESACOPLADO (célula seguinte), esse gargalo limita")
print("   somente as células de teste (N_test), NÃO o tamanho do treino — o S1")
print("   usa todo o seu pool. É o que evita o problema de 'treino com ~358 amostras'.\n")

print("Lado PT (costuma ser o limitante):")
for nome, S in [('S3 (PT/Eletrônicos)', S3_raw), ('S4 (PT/Beleza)', S4_raw)]:
    neg = int((S['label'] == 0).sum())
    pos = int((S['label'] == 1).sum())
    alerta = "  ⚠️ classe negativa baixa" if neg < 500 else ""
    print(f"  {nome}: negativos = {neg:,} | positivos = {pos:,}{alerta}")

### 2.6 Balanceamento e Split (Seção 5.4) — *treino desacoplado dos testes*

O balanceamento é feito em **duas etapas independentes**, para que um compartimento de teste escasso (tipicamente PT/Beleza na classe negativa) **não** rebaixe o tamanho do treino:

1. **Treino/validação:** o **S1** (EN/Eletrônicos) é balanceado no seu **próprio máximo** (`N_s1 = min(neg, pos)` *de S1*) e dividido 80/20 de forma estratificada. O treino usa **todo** o pool disponível.
2. **Matriz de teste:** as 4 células de avaliação (S1-val, S2, S3, S4) são balanceadas a um **N comum** (`N_test = min(neg, pos)` entre as *4 células de teste*), garantindo comparação justa do heatmap sem desperdiçar dados de treino.

> Isso corrige o acoplamento da regra original (`N = min` entre os 4 subconjuntos *antes* do split), que fazia o menor compartimento de teste limitar o treino inteiro.

In [ ]:
# Balanceamento DESACOPLADO: o treino usa todo o S1; os testes vão a um N comum.
from sklearn.model_selection import train_test_split

cols = ['id', 'idioma', 'dominio', 'label', 'texto']

def min_classe(S):
    return min(int((S['label']==0).sum()), int((S['label']==1).sum()))

def balancear(S, n, seed=SEED_DATA):
    pos = S[S['label']==1].sample(n=n, random_state=seed)
    neg = S[S['label']==0].sample(n=n, random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)

# (1) TREINO/VAL — S1 balanceado no SEU próprio máximo e dividido 80/20.
#     O tamanho do treino NÃO é mais rebaixado ao menor compartimento de teste.
N_s1 = min_classe(S1_raw)
assert N_s1 > 0, "S1 (treino) ficou sem exemplos de uma das classes."
S1_full = balancear(S1_raw, N_s1)[cols]
S1 = S1_full   # alias do S1 balanceado (usado na auditoria/estatística da Task 1.3)
S1_train, S1_val_full = train_test_split(
    S1_full, test_size=0.20, stratify=S1_full['label'], random_state=SEED_DATA)

# (2) MATRIZ DE TESTE — S1-val, S2, S3, S4 balanceadas a um N comum (comparação justa).
N_test = min(min_classe(S1_val_full), min_classe(S2_raw), min_classe(S3_raw), min_classe(S4_raw))
assert N_test > 0, "Algum compartimento de teste ficou sem exemplos de uma das classes."
S1_val = balancear(S1_val_full, N_test)[cols]
S2 = balancear(S2_raw, N_test)[cols]
S3 = balancear(S3_raw, N_test)[cols]
S4 = balancear(S4_raw, N_test)[cols]

print(f"TREINO  → S1_train: {len(S1_train):>6,} exemplos  (pool S1 com {N_s1:,} por classe, antes do split 80/20)")
print(f"VAL/TESTE → balanceados a N_test = {N_test:,} por classe ({2*N_test:,} cada):")
print(f"   T1 S1_val: {len(S1_val):>5,} | T2 S2: {len(S2):>5,} | T3 S3: {len(S3):>5,} | T4 S4: {len(S4):>5,}")
print(f"\nGanho vs. regra acoplada antiga: o treino agora tem {len(S1_train):,} exemplos "
      f"em vez de ~{2*N_test:,}.")

In [ ]:
# Salva os artefatos da Etapa 1 (split treino/val já foi feito na célula de balanceamento)
S1_train.to_csv(DIR_SAIDA/"S1_train_en_eletronicos.csv", index=False)
S1_val.to_csv(DIR_SAIDA/"S1_val_en_eletronicos.csv", index=False)
S2.to_csv(DIR_SAIDA/"S2_en_beleza.csv", index=False)
S3.to_csv(DIR_SAIDA/"S3_pt_eletronicos.csv", index=False)
S4.to_csv(DIR_SAIDA/"S4_pt_beleza.csv", index=False)

resumo = pd.DataFrame([
    ['S1 (treino)', 'EN', 'Eletrônicos', len(S1_train), 'Treino do modelo'],
    ['S1 (val)',    'EN', 'Eletrônicos', len(S1_val),   'Controle interno (T1)'],
    ['S2',          'EN', 'Beleza',      len(S2),       'Teste — Domain Shift (T2)'],
    ['S3',          'PT', 'Eletrônicos', len(S3),       'Teste — Language Shift (T3)'],
    ['S4',          'PT', 'Beleza',      len(S4),       'Teste — Combinado (T4)'],
], columns=['Subconjunto', 'Idioma', 'Domínio', 'N amostras', 'Função'])
print("Arquivos salvos em", DIR_SAIDA)
display(resumo)

### 2.7 Task 1.3 — Auditoria do Filtro EN via Classificação Zero-Shot (LLM)

A auditoria valida **apenas o lado EN (S1, S2)**, que é onde existe um filtro sujeito a erro (lexical). O lado **PT (S3, S4)** deriva da **categoria do produto** (`site_category_lv1`) — é ground-truth, então não é auditado por LLM (auditar ground-truth com um modelo mais fraco só injetaria ruído).

Para o EN, usamos um **classificador zero-shot** (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`) com rótulos **curtos e simétricos** (`"an electronics product"` × `"a beauty product"`). Rótulos longos/assimétricos (versão anterior) causavam **viés de comprimento** que inflava artificialmente a classe "beleza" — por isso eletrônicos apareciam com precisão baixa mesmo estando corretos.

**Justificativa metodológica:** o uso de LLMs como anotadores é prática consolidada em NLP
(Zhu et al., 2023 — *"Is GPT-4 a Good Data Annotator?"*), aceito desde que declarado na Metodologia.

**Pipeline:** amostra 100 textos de S1 e S2; classifica; compara com o domínio do filtro lexical;
calcula precisão (critério ≥ 80%); salva em `/content/data_processed/auditoria/`.

In [ ]:
# Task 1.3 — (1/3) Carregamento do classificador zero-shot (auditoria do filtro EN)
from transformers import pipeline
import torch

print("Carregando classificador zero-shot...")
print("(download ~550 MB na primeira execução — aguarde)")

zs_device = 0 if torch.cuda.is_available() else -1
classificador_zs = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=zs_device,
)
print(f"Modelo carregado. Rodando em: {'GPU' if zs_device == 0 else 'CPU'}")

# Rótulos CURTOS e SIMÉTRICOS — evitam o viés de comprimento que, com rótulos longos,
# inflava artificialmente a classe 'beleza'. Aplicados só sobre textos EN (S1, S2).
LABELS_ZS = ["an electronics product", "a beauty product"]
LABEL_ELETRONICOS = LABELS_ZS[0]
LABEL_BELEZA      = LABELS_ZS[1]
print("Labels zero-shot (balanceados):", LABELS_ZS)

In [ ]:
# Task 1.3 — (2/3) Anotação automática: 100 amostras dos subconjuntos EN (S1, S2)
# Auditamos SÓ o lado EN, que usa filtro lexical (sujeito a erro). O lado PT (S3, S4)
# vem da categoria do produto (site_category_lv1) -> ground-truth, não auditado por LLM.
import pandas as pd
from pathlib import Path

DIR_AUDIT = DIR_SAIDA / "auditoria"
DIR_AUDIT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16

def anotar_subconjunto(S, nome, lang, n=N_AUDITORIA, seed=SEED_DATA):
    amostra = S.sample(n=min(n, len(S)), random_state=seed).copy().reset_index(drop=True)
    textos  = amostra['texto'].str[:512].tolist()
    preditos = []
    for i in range(0, len(textos), BATCH_SIZE):
        batch  = textos[i : i + BATCH_SIZE]
        saidas = classificador_zs(batch, candidate_labels=LABELS_ZS, multi_label=False)
        for saida in saidas:
            label_top = saida['labels'][0]
            preditos.append('eletronicos' if label_top == LABEL_ELETRONICOS else 'beleza')
    amostra['dominio_llm'] = preditos
    amostra['correto']     = (amostra['dominio'] == amostra['dominio_llm']).astype(int)
    return amostra

resultados = {}
for S, nome, lang in [(S1,'S1','en'), (S2,'S2','en')]:
    print(f"Anotando {nome} [{lang}] — {min(N_AUDITORIA, len(S))} amostras...", end=' ')
    df_anot = anotar_subconjunto(S, nome, lang)
    resultados[nome] = df_anot
    caminho = DIR_AUDIT / f"auditoria_{nome}.csv"
    df_anot[['id','idioma','dominio','dominio_llm','correto','label','texto']].to_csv(caminho, index=False)
    prec = df_anot['correto'].mean()
    status = "✅" if prec >= LIMIAR_PRECISAO else "❌"
    print(f"precisão = {prec:.1%}  {status}")

print("\nLado PT (S3, S4): domínio por categoria — precisão 100% por construção (não auditado por LLM).")
print("Planilhas salvas em:", DIR_AUDIT)

In [ ]:
# Task 1.3 — (3/3) Estatísticas do filtro + Tabela de precisão final
import pandas as pd
from collections import Counter
import re
import matplotlib.pyplot as plt

# ---- EN: palavras-chave mais ativas (filtro lexical) ----
print("EN — palavras-chave mais frequentes (S1, S2):")
for S, nome in [(S1_raw,'S1'), (S2_raw,'S2')]:
    dom  = S['dominio'].iloc[0]
    kws  = KEYWORDS['en'][dom]
    cont = Counter()
    for t in S['texto'].str.lower():
        for kw in kws:
            if re.search(r"\b" + re.escape(kw) + r"\b", t):
                cont[kw] += 1
    top = ', '.join(f"{k} ({v})" for k, v in cont.most_common(5))
    print(f"  {nome} [en/{dom}]: {top}")

# ---- PT: categorias incluídas por domínio (site_category_lv1) ----
print("\nPT — categorias por domínio (site_category_lv1) — ground-truth, precisão 100%:")
for S, nome in [(S3_raw,'S3'), (S4_raw,'S4')]:
    dom  = S['dominio'].iloc[0]
    vc   = S['categoria'].value_counts()
    comp = ', '.join(f"{c} ({n:,})" for c, n in vc.items())
    print(f"  {nome} [pt/{dom}]: {comp}")

total_pt = len(df_pt)
atrib_pt = int(df_pt['dominio'].notna().sum())
print(f"\nSeletividade PT (por categoria): {atrib_pt:,}/{total_pt:,} "
      f"avaliações nos domínios escolhidos ({100*atrib_pt/total_pt:.1f}%).\n")

# ---- Tabela de precisão: EN por auditoria zero-shot; PT por categoria (ground-truth) ----
linhas = []
for nome, df_anot in resultados.items():
    prec   = df_anot['correto'].mean()
    status = "✅ aprovado" if prec >= LIMIAR_PRECISAO else "❌ refinar keywords"
    acord  = int(df_anot['correto'].sum())
    linhas.append([nome, str(len(df_anot)), str(acord), f"{prec:.1%}", status])
for nome in ['S3', 'S4']:
    linhas.append([nome, '—', '—', '100% (categoria)', '✅ ground-truth'])

tabela = pd.DataFrame(linhas, columns=[
    'Subconjunto', 'N amostrado', 'Acordos (filtro=LLM)', 'Precisão', 'Critério ≥ 80%'
])
print("Precisão da filtragem — EN por auditoria zero-shot; PT por categoria (ground-truth):")
display(tabela)

# ---- Gráfico: precisão dos subconjuntos EN auditados ----
aud = pd.DataFrame([[nome, df_anot['correto'].mean()] for nome, df_anot in resultados.items()],
                   columns=['Subconjunto', 'prec'])
fig, ax = plt.subplots(figsize=(5.5, 3.5))
cores = ['#2ecc71' if p >= LIMIAR_PRECISAO else '#e74c3c' for p in aud['prec']]
barras = ax.bar(aud['Subconjunto'], aud['prec'], color=cores, edgecolor='white')
ax.axhline(LIMIAR_PRECISAO, color='gray', linestyle='--', linewidth=1, label='Limiar 80%')
ax.set_ylim(0, 1.05); ax.set_ylabel('Precisão do filtro EN')
ax.set_title('Auditoria zero-shot — filtro lexical EN (S1, S2)')
for bar, p in zip(barras, aud['prec']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f"{p:.1%}",
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

falhas = aud[aud['prec'] < LIMIAR_PRECISAO]
if not falhas.empty:
    print("\n⚠️  EN abaixo de 80%:", list(falhas['Subconjunto']), "→ refinar keywords EN.")
else:
    print("\n✅ Filtro EN aprovado (≥ 80%). PT é ground-truth por categoria.")

### 🔄 Justificativa: Escolha dos Domínios e Método de Filtragem

A definição passou por iterações até a configuração atual — **Eletrônicos × Beleza**, com filtragem **EN por keyword** e **PT por categoria**:

| Critério | ① Produto × Logística | ② Eletrônicos × Livros | ③ **Eletrônicos × Beleza (atual)** |
|---|---|---|---|
| Sobreposição de vocabulário | Alta | Mínima | Mínima |
| Volume da classe negativa em PT | OK | Insuficiente (livros raros) ❌ | **2.372** (Beleza e Perfumaria) ✅ |
| Método de filtro PT | keyword | keyword | **categoria (site_category_lv1)** |
| Precisão do filtro PT | ~52% (logística) | keyword | **~100% (ground-truth)** |

**As duas correções que destravaram o projeto:**
1. **Balanceamento desacoplado** (Seção 2.6): o treino (S1) deixou de ser rebaixado ao menor compartimento de teste.
2. **Filtragem PT por categoria** (Etapa 0): o filtro por keyword capturava só ~300 negativos de beleza; a categoria `Beleza e Perfumaria` tem **2.372**. O gargalo era o método de filtro, não o domínio.

> Usar metadado de categoria (PT) é metodologicamente superior ao filtro lexical, e usar categorias de produto como domínios alinha com a literatura canônica de domain adaptation (Blitzer et al., 2007). O lado EN permanece por keyword porque o `amazon_polarity` não tem categoria — assimetria declarada e validada pela auditoria zero-shot.

### 2.9 Consolidação Modular — `src/data_pipeline.py`

A Task 1.1 do plano (`PLAN-congelamento-transformers.md`) lista explicitamente como **OUTPUT** o script `src/data_pipeline.py`. As células abaixo materializam esse entregável: encapsulam a lógica de download, filtragem, particionamento e balanceamento (já validada nas seções anteriores) em um módulo Python reutilizável que será importado pelas Etapas 2–4 e versionado no repositório do projeto.

In [ ]:
# Gera o módulo src/data_pipeline.py a partir da lógica já validada nas seções 2.2–2.6
from pathlib import Path

DIR_SRC = Path('/content/src')
DIR_SRC.mkdir(parents=True, exist_ok=True)
(DIR_SRC / '__init__.py').write_text('', encoding='utf-8')

MODULO_DATA_PIPELINE = r'''# -*- coding: utf-8 -*-
"""
src/data_pipeline.py
====================
Módulo de preparação de dados da Etapa 1 do projeto
"Análise Arquitetural do Congelamento de Camadas em Transformers Multilíngues".

Atribuição de domínio HÍBRIDA:
  * EN (amazon_polarity, sem categoria) -> filtro lexical por palavra-chave (lista enxuta).
  * PT (B2W, com site_category_lv1)     -> categoria do produto (ground-truth).
"""
from __future__ import annotations

import os
import re
import itertools
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd

# ============================================================
# Domínios (Seção 5.2) — EN por keyword, PT por categoria
# ============================================================
KEYWORDS: Dict[str, Dict[str, List[str]]] = {
    "en": {
        "eletronicos": ["battery","usb","charger","charging","wifi","bluetooth",
                        "smartphone","laptop","tablet","touchscreen","headphone",
                        "headphones","earbuds","smartwatch","phone","phones","router","hdmi"],
        "beleza":      ["skin","scent","fragrance","perfume","cream",
                        "lotion","shampoo","conditioner","moisturizer","makeup",
                        "cosmetic","serum","sunscreen","soap","wrinkle"],
    },
}

CATEGORIAS_PT: Dict[str, str] = {
    "Celulares e Smartphones":  "eletronicos",
    "Informática e Acessórios": "eletronicos",
    "TV e Home Theater":        "eletronicos",
    "Beleza e Perfumaria":      "beleza",
}

URL_B2W = "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv"

# ============================================================
# Filtro lexical (EN)
# ============================================================
def _compilar_regex(palavras: List[str]) -> re.Pattern:
    partes = [re.escape(p.lower()) for p in palavras]
    return re.compile(r"\b(?:" + "|".join(partes) + r")\b", re.UNICODE)

REGEX = {lang: {dom: _compilar_regex(kws) for dom, kws in doms.items()}
         for lang, doms in KEYWORDS.items()}


def classificar_dominio(texto: str, lang: str = "en") -> Optional[str]:
    """EN: >=1 keyword do domínio E nenhuma do outro. Retorna 'eletronicos'|'beleza'|None."""
    if not isinstance(texto, str) or not texto.strip():
        return None
    t = texto.lower()
    tem_eletr  = bool(REGEX[lang]["eletronicos"].search(t))
    tem_beleza = bool(REGEX[lang]["beleza"].search(t))
    if tem_eletr and not tem_beleza:
        return "eletronicos"
    if tem_beleza and not tem_eletr:
        return "beleza"
    return None


def categoria_para_dominio(cat) -> Optional[str]:
    """PT: mapeia site_category_lv1 -> 'eletronicos'|'beleza'|None."""
    if not isinstance(cat, str):
        return None
    return CATEGORIAS_PT.get(cat.strip(), None)


# ============================================================
# Mapeamento estrela -> sentimento (Seção 4)
# ============================================================
def estrela_para_sentimento(nota) -> Optional[int]:
    """1-2 -> 0 (Negativo) | 4-5 -> 1 (Positivo) | 3 -> None (descartado)."""
    try:
        n = float(nota)
    except (TypeError, ValueError):
        return None
    if n <= 2: return 0
    if n >= 4: return 1
    return None


# ============================================================
# Carregamento (Task 1.1)
# ============================================================
def carregar_amazon_en_streaming(
    max_candidatos_por_dominio: int = 20000,
    max_linhas: int = 600000,
) -> pd.DataFrame:
    """Lê amazon_polarity em streaming e coleta candidatos por domínio (keyword)."""
    from datasets import load_dataset

    stream = None
    erros = []
    for repo in ["fancyzhx/amazon_polarity", "amazon_polarity"]:
        try:
            stream = load_dataset(repo, split="train", streaming=True)
            break
        except Exception as e:
            erros.append(f"{repo}: {e}")
    if stream is None:
        raise RuntimeError("Falha ao carregar Amazon EN:\n" + "\n".join(erros))

    coletado = {"eletronicos": [], "beleza": []}
    vistos = 0
    for ex in stream:
        vistos += 1
        texto = ((ex.get("title") or "") + ". " + (ex.get("content") or "")).strip()
        dom = classificar_dominio(texto, "en")
        if dom is not None and len(coletado[dom]) < max_candidatos_por_dominio:
            coletado[dom].append({
                "id": f"amz_{vistos}", "texto": texto,
                "label": int(ex["label"]), "idioma": "en", "dominio": dom,
            })
        if vistos >= max_linhas: break
        if (len(coletado["eletronicos"]) >= max_candidatos_por_dominio and
            len(coletado["beleza"]) >= max_candidatos_por_dominio): break
    return pd.DataFrame(coletado["eletronicos"] + coletado["beleza"])


def carregar_b2w(caminho: str = "/content/b2w.csv") -> pd.DataFrame:
    """Lê o B2W; baixa do repositório oficial se o arquivo não existir."""
    if os.path.exists(caminho):
        return pd.read_csv(caminho, low_memory=False)
    df = pd.read_csv(URL_B2W, low_memory=False)
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(caminho, index=False)
    return df


def _col(df: pd.DataFrame, *nomes) -> Optional[str]:
    for n in nomes:
        if n in df.columns: return n
    return None


def _serie(df: pd.DataFrame, nome: Optional[str]) -> pd.Series:
    if nome is None:
        return pd.Series([""] * len(df))
    return df[nome].fillna("").astype(str)


def normalizar_b2w_pt(df_b2w: pd.DataFrame) -> pd.DataFrame:
    """Normaliza B2W em [id, texto, nota, label, idioma, categoria, dominio].

    O domínio vem de site_category_lv1 (categoria do produto), não de keyword.
    """
    b = pd.DataFrame()
    b["texto"] = (_serie(df_b2w, _col(df_b2w, "review_title")) + ". " +
                  _serie(df_b2w, _col(df_b2w, "review_text"))).str.strip()
    c_nota = _col(df_b2w, "overall_rating")
    b["nota"] = pd.to_numeric(df_b2w[c_nota], errors="coerce") if c_nota else None
    b["categoria"] = _serie(df_b2w, _col(df_b2w, "site_category_lv1"))
    b["label"] = b["nota"].apply(estrela_para_sentimento)
    b = b[b["texto"].str.len() > 0]
    b = b.dropna(subset=["label"]).copy()
    b["label"]   = b["label"].astype(int)
    b["idioma"]  = "pt"
    b["id"]      = ["pt_" + str(i) for i in range(len(b))]
    b["dominio"] = b["categoria"].apply(categoria_para_dominio)
    return b


# ============================================================
# Particionamento (Task 1.2)
# ============================================================
def construir_particoes(df_en: pd.DataFrame, df_pt: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Aplica a partição S1..S4 sobre os pools rotulados e VERIFICA a disjunção de IDs."""
    S1 = df_en[df_en["dominio"] == "eletronicos"].copy()  # EN Eletrônicos (keyword)
    S2 = df_en[df_en["dominio"] == "beleza"].copy()        # EN Beleza (keyword)
    S3 = df_pt[df_pt["dominio"] == "eletronicos"].copy()  # PT Eletrônicos (categoria)
    S4 = df_pt[df_pt["dominio"] == "beleza"].copy()        # PT Beleza (categoria)

    ids = {"S1": set(S1["id"]), "S2": set(S2["id"]),
           "S3": set(S3["id"]), "S4": set(S4["id"])}
    for a, b in itertools.combinations(ids, 2):
        if ids[a] & ids[b]:
            raise AssertionError(f"Interseção de IDs entre {a} e {b}")
    return {"S1": S1, "S2": S2, "S3": S3, "S4": S4}


# ============================================================
# Balanceamento e split (Seção 5.4) — treino DESACOPLADO dos testes
# ============================================================
def _min_classe(S: pd.DataFrame) -> int:
    return min(int((S["label"] == 0).sum()), int((S["label"] == 1).sum()))


def balancear(S: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    pos = S[S["label"] == 1].sample(n=n, random_state=seed)
    neg = S[S["label"] == 0].sample(n=n, random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)


def balancear_e_dividir(
    particoes: Dict[str, pd.DataFrame], seed: int = 42, test_size: float = 0.2,
) -> Dict[str, pd.DataFrame]:
    """Treino (S1) balanceado no próprio máximo e dividido 80/20; testes a um N comum."""
    from sklearn.model_selection import train_test_split

    cols = ["id", "idioma", "dominio", "label", "texto"]

    N_s1 = _min_classe(particoes["S1"])
    if N_s1 <= 0:
        raise ValueError("S1 (treino) ficou sem exemplos de uma das classes.")
    S1_full = balancear(particoes["S1"], N_s1, seed=seed)[cols]
    S1_train, S1_val_full = train_test_split(
        S1_full, test_size=test_size, stratify=S1_full["label"], random_state=seed)

    N_test = min(_min_classe(S1_val_full),
                 _min_classe(particoes["S2"]),
                 _min_classe(particoes["S3"]),
                 _min_classe(particoes["S4"]))
    if N_test <= 0:
        raise ValueError("Algum compartimento de teste ficou sem exemplos de uma das classes.")

    return {
        "S1_train": S1_train,
        "S1_val": balancear(S1_val_full, N_test, seed=seed)[cols],
        "S2": balancear(particoes["S2"], N_test, seed=seed)[cols],
        "S3": balancear(particoes["S3"], N_test, seed=seed)[cols],
        "S4": balancear(particoes["S4"], N_test, seed=seed)[cols],
        "N_s1": N_s1, "N_test": N_test,
    }


# ============================================================
# Orquestrador
# ============================================================
def preparar_etapa1(
    dir_saida: str = "/content/data_processed",
    caminho_b2w: str = "/content/b2w.csv",
    max_candidatos_en: int = 20000,
    max_linhas_amazon: int = 600000,
    seed: int = 42,
) -> Dict[str, pd.DataFrame]:
    """Executa Tasks 1.1 e 1.2 ponta a ponta e salva os 5 CSVs em `dir_saida`."""
    dir_saida = Path(dir_saida); dir_saida.mkdir(parents=True, exist_ok=True)

    df_en = carregar_amazon_en_streaming(max_candidatos_en, max_linhas_amazon)
    df_pt = normalizar_b2w_pt(carregar_b2w(caminho_b2w))
    particoes = construir_particoes(df_en, df_pt)
    artefatos = balancear_e_dividir(particoes, seed=seed)

    artefatos["S1_train"].to_csv(dir_saida/"S1_train_en_eletronicos.csv", index=False)
    artefatos["S1_val"  ].to_csv(dir_saida/"S1_val_en_eletronicos.csv",   index=False)
    artefatos["S2"      ].to_csv(dir_saida/"S2_en_beleza.csv",        index=False)
    artefatos["S3"      ].to_csv(dir_saida/"S3_pt_eletronicos.csv",   index=False)
    artefatos["S4"      ].to_csv(dir_saida/"S4_pt_beleza.csv",        index=False)
    return artefatos


# ============================================================
# Auto-teste minimalista
# ============================================================
if __name__ == "__main__":
    assert classificar_dominio("the battery life is amazing", "en") == "eletronicos"
    assert classificar_dominio("this moisturizer cream is great for my skin", "en") == "beleza"
    assert classificar_dominio("the laptop smells like perfume", "en") is None
    assert categoria_para_dominio("Celulares e Smartphones") == "eletronicos"
    assert categoria_para_dominio("Beleza e Perfumaria") == "beleza"
    assert categoria_para_dominio("Livros") is None
    assert estrela_para_sentimento(1) == 0
    assert estrela_para_sentimento(5) == 1
    assert estrela_para_sentimento(3) is None
    print("data_pipeline OK")
'''

destino = DIR_SRC / 'data_pipeline.py'
destino.write_text(MODULO_DATA_PIPELINE, encoding='utf-8')
print(f'Módulo gerado: {destino} ({destino.stat().st_size:,} bytes)')

In [ ]:
# Smoke test do módulo: importa e exercita as funções públicas
import sys, importlib
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
for mod in ('src.data_pipeline', 'src'):
    if mod in sys.modules: del sys.modules[mod]

from src import data_pipeline as dp

# EN por keyword
assert dp.classificar_dominio("the battery life is amazing", "en") == "eletronicos"
assert dp.classificar_dominio("this moisturizer cream is great for my skin", "en") == "beleza"
assert dp.classificar_dominio("the laptop smells like perfume", "en") is None
# PT por categoria
assert dp.categoria_para_dominio("Celulares e Smartphones") == "eletronicos"
assert dp.categoria_para_dominio("Beleza e Perfumaria") == "beleza"
assert dp.categoria_para_dominio("Livros") is None
print("✅ classificação (EN keyword / PT categoria)")

assert dp.estrela_para_sentimento(1) == 0
assert dp.estrela_para_sentimento(5) == 1
assert dp.estrela_para_sentimento(3) is None
print("✅ estrela_para_sentimento")

particoes = dp.construir_particoes(df_en, df_pt)
assert set(particoes) == {"S1", "S2", "S3", "S4"}
print(f"✅ construir_particoes — S1={len(particoes['S1']):,} S2={len(particoes['S2']):,} "
      f"S3={len(particoes['S3']):,} S4={len(particoes['S4']):,}")

artefatos = dp.balancear_e_dividir(particoes, seed=SEED_DATA)
n_test = artefatos["N_test"]
assert len(artefatos["S1_val"]) == len(artefatos["S2"]) == len(artefatos["S3"]) == len(artefatos["S4"]) == 2*n_test
assert len(artefatos["S1_train"]) >= len(artefatos["S2"])
print(f"✅ balancear_e_dividir — treino={len(artefatos['S1_train']):,} | "
      f"N_test por classe={n_test:,} | células de teste={2*n_test:,} cada")

print("\n🎯 Módulo src/data_pipeline.py validado — pronto para uso nas Etapas 2–4.")

### ✅ 2.8 Resumo da Etapa 1 — *Checklist* de Verificação

| Critério (do plano) | Status |
|---------------------|--------|
| **Task 1.1** — datasets carregados (Amazon EN via streaming + B2W PT) e explorados | ✔️ automatizado |
| **Task 1.1 OUTPUT** — módulo `src/data_pipeline.py` gerado e testado | ✔️ seção 2.9 |
| **Task 1.2** — partições S1–S4 (EN: keyword · PT: categoria `site_category_lv1`) | ✔️ executado |
| **Task 1.2 VERIFY** — partições sem interseção de IDs | ✔️ `assert` na célula 2.4 + no módulo |
| Mapeamento estrela → sentimento binário (1–2 neg, 4–5 pos, 3 descartado) | ✔️ |
| Balanceamento desacoplado (treino no máx. do S1; testes em N comum) + split | ✔️ |
| Artefatos salvos em `/content/data_processed/` | ✔️ 5 CSVs |
| **Task 1.3** — auditoria de qualidade via classificação zero-shot (LLM) | ✔️ automatizado |

> **Nota metodológica:** domínios em PT vêm da categoria do produto (`site_category_lv1`,
> precisão ~100%); em EN, do filtro lexical (validado pela auditoria zero-shot). A assimetria
> é declarada na Metodologia (cf. Zhu et al., 2023 para o uso de LLM como anotador).

> **Próximo passo:** **Etapa 2** — arquitetura do classificador XLM-RoBERTa e congelamento C1–C4.

## 3. Resultados
*(A ser preenchido a partir da Etapa 4 — avaliação Zero-Shot nas 4 células, F1-macro, Δ-shift, heatmaps e curvas de perda.)*

## 4. Conclusão
*(A ser preenchido — suporte ou não às hipóteses H1 e H2 e discussão final.)*